In [ ]:
import os
DATA_DIR = os.getenv('DATA_DIR', '/path/to/local/private/data')
SYS_ORDER_FILE = os.getenv('SYS_ORDER_FILE', 'private_sys_order.xlsx')
GEO_FILE = os.getenv('GEO_FILE', 'private_geo.xlsx')
GOOGLE_MAPS_KEY = os.getenv('GOOGLE_MAPS_KEY', '')


In [ ]:
# Install necessary packages
%pip install pandas openpyxl matplotlib seaborn geopy numpy pyproj ace_tools math folium


##### Load in the Data Set
###### Note: Ensure the notebook and Excel files are in the same directory.
###### Open the notebook in VS Code.
###### Run the first cell to install the required packages.
###### Run the subsequent cells to load and display the data.


In [ ]:
# Importing the necessary libraries
import pandas as pd

# Function to load datasets
def load_data():
    # Load the datasets from the provided file paths
    order_data_path = os.path.join(DATA_DIR, SYS_ORDER_FILE)
    geo_data_path = os.path.join(DATA_DIR, GEO_FILE)
    
    # Read the Excel files
    order_data = pd.read_excel(order_data_path, sheet_name=None)
    geo_data = pd.read_excel(geo_data_path)
    
    # Accessing specific sheets in <private-order-file>.xlsx
    order_header = order_data['order_header']
    order_line = order_data['order_line']
    
    return order_header, order_line, geo_data

# Load the data
order_header, order_line, geo_data = load_data()

# Display the first few rows of each dataframe to ensure correct loading
print(order_header.head())



In [ ]:
print(order_line.head())


In [ ]:
print(geo_data.head())

##### Clean Data:

###### Handle missing values and ensure data types are correct.


In [ ]:
# Clean the data: handle missing values and correct data types
def clean_data(order_header, order_line, geo_data):
    # Check for missing values and fill or drop them as necessary
    order_header = order_header.dropna(subset=['status', 'consignment', 'ship_by_date', 'is_vas', 'carrier_id', 'customer_id_slim'])
    order_line = order_line.dropna(subset=['order_id', 'full_pallets'])
    geo_data = geo_data.dropna()
    
    # Ensure correct data types using .loc to avoid SettingWithCopyWarning
    order_header.loc[:, 'ship_by_date'] = pd.to_datetime(order_header['ship_by_date'])
    
    return order_header, order_line, geo_data

# Clean the data
order_header, order_line, geo_data = clean_data(order_header, order_line, geo_data)

# Display cleaned data info
print("Cleaned Order Header Data Info:")
print(order_header.info())



In [ ]:
print("\nCleaned Order Line Data Info:")
print(order_line.info())


In [ ]:

print("\nCleaned Geo Data Info:")
print(geo_data.info())

##### Additional Data Cleaning 
###### Remove Duplicates: Ensure there are no duplicate entries in the datasets.
###### Validate Values: Ensure that values fall within expected ranges or categories.
###### Handle Inconsistent Data: Standardize formats for columns like postcode and customer_id.

In [ ]:
# Step 1: Remove duplicates from the datasets
def remove_duplicates(order_header, order_line, geo_data):
    order_header = order_header.drop_duplicates()
    order_line = order_line.drop_duplicates()
    geo_data = geo_data.drop_duplicates()
    return order_header, order_line, geo_data

# Apply step 1
order_header, order_line, geo_data = remove_duplicates(order_header, order_line, geo_data)

# Display cleaned data info after removing duplicates
print("Order Header Data Info after Removing Duplicates:")
print(order_header.info())

print("\nOrder Line Data Info after Removing Duplicates:")
print(order_line.info())

print("\nGeo Data Info after Removing Duplicates:")
print(geo_data.info())


In [ ]:
# Step 2: Validate values in the datasets
def validate_values(order_header, order_line, geo_data):
    # Example validation: ensuring 'full_pallets' is non-negative
    order_line = order_line[order_line['full_pallets'] >= 0]
    return order_header, order_line, geo_data

# Apply step 2
order_header, order_line, geo_data = validate_values(order_header, order_line, geo_data)

# Display cleaned data info after validating values
print("Order Header Data Info after Validating Values:")
print(order_header.info())

print("\nOrder Line Data Info after Validating Values:")
print(order_line.info())

print("\nGeo Data Info after Validating Values:")
print(geo_data.info())


In [ ]:
# Step 3: Handle inconsistent data formats
def standardize_formats(order_header, order_line, geo_data):
    # Standardize formats for 'postcode' and 'customer_id'
    order_header['postcode'] = order_header['postcode'].str.upper().str.replace(' ', '')
    order_line['customer_id'] = order_line['customer_id'].str.upper().str.strip()
    geo_data['POSTCODE'] = geo_data['POSTCODE'].str.upper().str.replace(' ', '')
    geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.upper().str.strip()
    return order_header, order_line, geo_data

# Apply step 3
order_header, order_line, geo_data = standardize_formats(order_header, order_line, geo_data)

# Display cleaned data info after standardizing formats
print("Order Header Data Info after Standardizing Formats:")
print(order_header.info())

print("\nOrder Line Data Info after Standardizing Formats:")
print(order_line.info())

print("\nGeo Data Info after Standardizing Formats:")
print(geo_data.info())


##### Visualization 
###### Plot Distribution of Key Columns: Plot histograms or box plots for numerical columns to understand their distributions.
###### Bar Plots for Categorical Data: Visualize the frequency of categories in key categorical columns.
###### Scatter Plots for Relationships: Use scatter plots to identify relationships between key numerical columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to visualize data distributions and relationships
def visualize_data(order_header, order_line, geo_data):
    # Set up the matplotlib figure
    plt.figure(figsize=(14, 7))
    
    # Plot distribution of full_pallets in order_line
    plt.subplot(2, 2, 1)
    sns.histplot(order_line['full_pallets'], kde=True)
    plt.title('Distribution of Full Pallets')
    
    # Plot distribution of ship_by_date in order_header
    plt.subplot(2, 2, 2)
    sns.histplot(order_header['ship_by_date'], kde=True, bins=30)
    plt.title('Distribution of Ship By Date')
    
    # Plot count of statuses in order_header
    plt.subplot(2, 2, 3)
    sns.countplot(y=order_header['status'])
    plt.title('Count of Statuses in Order Header')
    
    # Plot count of carriers in order_header
    plt.subplot(2, 2, 4)
    sns.countplot(y=order_header['carrier_id'])
    plt.title('Count of Carriers in Order Header')
    
    plt.tight_layout()
    plt.show()

# Visualize the data
visualize_data(order_header, order_line, geo_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to visualize data distributions and relationships
def visualize_data(order_header, order_line, geo_data):
    # Set up the matplotlib figure
    plt.figure(figsize=(14, 7))
    
    # Plot distribution of full_pallets in order_line (log scale)
    plt.subplot(2, 2, 1)
    sns.histplot(order_line['full_pallets'], kde=True)
    plt.yscale('log')
    plt.title('Distribution of Full Pallets (Log Scale)')
    
    # Plot distribution of ship_by_date in order_header with different line color and horizontal scale in months
    plt.subplot(2, 2, 2)
    sns.histplot(order_header['ship_by_date'], kde=True, bins=30)
    plt.title('Distribution of Ship By Date')
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
    plt.xlabel('Ship By Date 2024')
    
    # Plot count of statuses in order_header (unchanged)
    plt.subplot(2, 2, 3)
    sns.countplot(y=order_header['status'])
    plt.title('Count of Statuses in Order Header')
    
    # Plot count of carriers in order_header with adjusted vertical scale (log scale)
    plt.subplot(2, 2, 4)
    sns.countplot(x=order_header['carrier_id'])
    plt.yscale('log')
    plt.title('Count of Carriers in Order Header')
    plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
    
    plt.tight_layout()
    plt.show()

# Visualize the data
visualize_data(order_header, order_line, geo_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to create various plots
def create_plots(order_header, order_line, geo_data):
    # Set up the matplotlib figure
    plt.figure(figsize=(14, 10))
    
    # Bar plot: distribution of carriers by full pallets
    plt.subplot(3, 2, 1)
    sns.barplot(x=order_header['carrier_id'], y=order_line['full_pallets'], errorbar=None)
    plt.title('Distribution of Full Pallets by Carrier')
    plt.xlabel('Carrier ID')
    plt.ylabel('Full Pallets')
    plt.xticks(rotation=90)
    
    # Bar plot: count of shipments by status
    plt.subplot(3, 2, 2)
    sns.countplot(x=order_header['status'])
    plt.title('Count of Shipments by Status')
    plt.xlabel('Status')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    
    # Heatmap: correlation matrix of numeric features
    plt.subplot(3, 2, 3)
    numeric_features = order_line[['expected_weight', 'full_pallets', 'cpl', 'lpp', 'cpp', 'pallet_type']]
    correlation_matrix = numeric_features.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Correlation Matrix of Numeric Features')
    
    # Box plot: distribution of full pallets by status
    plt.subplot(3, 2, 4)
    sns.boxplot(x=order_header['status'], y=order_line['full_pallets'])
    plt.title('Distribution of Full Pallets by Status')
    plt.xlabel('Status')
    plt.ylabel('Full Pallets')
    plt.xticks(rotation=45)
    
    # Scatter plot: full_pallets vs. ship_by_date with horizontal scale in months
    plt.subplot(3, 2, 5)
    sns.scatterplot(x=order_header['ship_by_date'], y=order_line['full_pallets'], alpha=0.6, edgecolor='w', s=40)
    plt.title('Full Pallets vs. Ship By Date')
    plt.xlabel('2024')
    plt.ylabel('Full Pallets')
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
    
    plt.tight_layout()
    plt.show()

# Generate the plots
create_plots(order_header, order_line, geo_data)


##### Filter Orders Based on Given Criteria:
###### Status: 'Released'
###### Consignment starts with 'Q'
###### Ship by Date: > current time + 4 hours
###### Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']
###### is_vas is 'N'
###### Sum of full pallets per consignment < 45
###### Same customer ID for both load opportunities
###### Sum of full pallets on both load opportunities <= 52

In [ ]:
# Assuming order_header, order_line, geo_data have already been loaded and cleaned

# Step 1: Filter orders based on status 'Released'
def filter_by_status(order_header):
    return order_header[order_header['status'] == 'Released']

# Apply the first filter
filtered_order_header = filter_by_status(order_header)
print("Filtered by Status 'Released':")
print(filtered_order_header.head())


In [ ]:
# Step 2: Filter orders where consignment starts with 'Q'
def filter_by_consignment(order_header):
    return order_header[order_header['consignment'].str.startswith('Q')]

# Apply the second filter
filtered_order_header = filter_by_consignment(filtered_order_header)
print("Filtered by Consignment starting with 'Q':")
print(filtered_order_header.head())


In [ ]:
from datetime import datetime, timedelta

# Step 3: Filter orders with ship_by_date > current time + 4 hours
def filter_by_ship_date(order_header):
    current_time_plus_4_hours = datetime.now() + timedelta(hours=4)
    return order_header[order_header['ship_by_date'] > current_time_plus_4_hours]

# Apply the third filter
filtered_order_header = filter_by_ship_date(filtered_order_header)
print("Filtered by Ship by Date > Current Time + 4 Hours:")
print(filtered_order_header.head())


In [ ]:
# Step 4: Filter orders where is_vas is 'N'
def filter_by_is_vas(order_header):
    return order_header[order_header['is_vas'] == 'N']

# Apply the fourth filter
filtered_order_header = filter_by_is_vas(filtered_order_header)
print("Filtered by is_vas 'N':")
print(filtered_order_header.head())


In [ ]:
# Step 5: Filter orders with carrier_id not in the specified list
def filter_by_carrier_id(order_header):
    excluded_carriers = ['SDS', 'DHL', 'CCO', 'MRT']
    return order_header[~order_header['carrier_id'].isin(excluded_carriers)]

# Apply the fifth filter
filtered_order_header = filter_by_carrier_id(filtered_order_header)
print("Filtered by Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']:")
print(filtered_order_header.head())


In [ ]:
# Step 6: Filter consignments with sum of full pallets per consignment < 45
def filter_by_pallet_sums(order_header, order_line):
    merged_data = pd.merge(order_header, order_line, on='order_id')
    consignment_pallet_sums = merged_data.groupby('consignment')['full_pallets'].sum().reset_index()
    return consignment_pallet_sums[consignment_pallet_sums['full_pallets'] < 45]

# Apply the sixth filter
filtered_consignments = filter_by_pallet_sums(filtered_order_header, order_line)
print("Filtered Consignments with Sum of Full Pallets < 45:")
print(filtered_consignments.head())


In [ ]:
# Function to identify consolidation opportunities based on filtered consignments
def identify_consolidation_opportunities(filtered_order_header, filtered_consignments, order_line):
    potential_opportunities = pd.merge(filtered_consignments, filtered_order_header, on='consignment')
    # Ensure only the numeric 'full_pallets' column is summed
    consolidation_opportunities = potential_opportunities.groupby(['customer_id_slim', 'consignment'])[['full_pallets']].sum().reset_index()
    return consolidation_opportunities[consolidation_opportunities['full_pallets'] <= 52]

# Apply the seventh filter and identify consolidation opportunities
filtered_order_header_final = filtered_order_header[filtered_order_header['consignment'].isin(filtered_consignments['consignment'])]
consolidation_opportunities = identify_consolidation_opportunities(filtered_order_header_final, filtered_consignments, order_line)
print("\nConsolidation Opportunities:")
print(consolidation_opportunities)


In [ ]:
# Load geo_data if not already loaded
geo_data_path = os.path.join(DATA_DIR, GEO_FILE)
geo_data = pd.read_excel(geo_data_path)


In [ ]:
import pandas as pd

# Load order data
order_data = pd.read_excel(os.path.join(DATA_DIR, SYS_ORDER_FILE), sheet_name=None)
order_header = order_data['order_header']
order_line = order_data['order_line']

# Load geo data
geo_data = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# Standardize customer ID format in geo data
geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.replace('-', '')

# Display the first few rows of each dataframe to understand the structure
print("Order Header:")
print(order_header.head())
print("\nOrder Line:")
print(order_line.head())
print("\nGeo Data:")
print(geo_data.head())


In [ ]:
import pandas as pd
from math import radians, cos, sin, sqrt, atan2
import pyproj

# Load order data
order_data = pd.read_excel(os.path.join(DATA_DIR, SYS_ORDER_FILE), sheet_name=None)
order_header = order_data['order_header']
order_line = order_data['order_line']

# Load geo data
geo_data = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# Standardize customer ID format in geo data
geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.replace('-', '')

# Merge order header and order line data on order_id
order_details = order_header.merge(order_line, on='order_id')

# Calculate total full pallets per order
order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')

# Merge with geo data
order_details['customer_id_slim_x'] = order_details['customer_id_slim_x'].astype(str)
merged_orders = order_details.merge(geo_data, left_on='customer_id_slim_x', right_on='CUSTOMER_ID', how='left')

# Function to convert easting/northing coordinates to latitude/longitude
def easting_northing_to_lat_lon(easting, northing):
    transformer = pyproj.Transformer.from_crs('epsg:27700', 'epsg:4326')  # British National Grid to WGS84
    lat, lon = transformer.transform(easting, northing)
    return lat, lon

# Add latitude and longitude columns
merged_orders[['latitude', 'longitude']] = merged_orders.apply(
    lambda row: easting_northing_to_lat_lon(row['HOME EASTING'], row['HOME NORTHING']), axis=1, result_type='expand'
)

# Drop rows with missing or invalid latitude/longitude values
merged_orders = merged_orders.dropna(subset=['latitude', 'longitude'])
merged_orders = merged_orders[(merged_orders['latitude'] >= -90) & (merged_orders['latitude'] <= 90)]
merged_orders = merged_orders[(merged_orders['longitude'] >= -180) & (merged_orders['longitude'] <= 180)]

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# Identify nearby consolidation opportunities
opportunities = []
for i, row1 in merged_orders.iterrows():
    for j, row2 in merged_orders.iterrows():
        if i >= j:
            continue
        # Ensure valid coordinates
        if pd.notna(row1['latitude']) and pd.notna(row1['longitude']) and pd.notna(row2['latitude']) and pd.notna(row2['longitude']):
            distance = haversine_distance(row1['latitude'], row1['longitude'], row2['latitude'], row2['longitude'])
            if distance <= 48:  # 30 miles is approximately 48 kilometers
                combined_pallets = row1['total_full_pallets'] + row2['total_full_pallets']
                opportunities.append((row1['order_id'], row2['order_id'], combined_pallets, distance))

# Convert opportunities to DataFrame for better visualization
opportunities_df = pd.DataFrame(opportunities, columns=['Order 1', 'Order 2', 'Combined Pallets', 'Distance (km)'])

# Group by 'Order 1' and 'Order 2' to remove duplicate pairs
opportunities_df = opportunities_df.groupby(['Order 1', 'Order 2']).agg({'Combined Pallets': 'first', 'Distance (km)': 'first'}).reset_index()

# Save the opportunities DataFrame to an Excel file for easy review
opportunities_df.to_excel(os.path.join(DATA_DIR, 'consolidation_opportunities_coordinates_postcodes.xlsx'), index=False)

print("Consolidation opportunities have been saved to os.path.join(DATA_DIR, 'consolidation_opportunities_coordinates_postcodes.xlsx').")


##### Approach 2: Using Google Maps API
###### Geocoding:
###### Postcodes are geocoded using the Google Maps API to obtain latitude/longitude coordinates.
###### This approach provides accurate geolocation data, leveraging Google's robust mapping service.
###### Distance Calculation and Opportunity Identification:
###### Similar to the first approach, the haversine formula is used for distance calculation.
###### Opportunities are identified by iterating through pairs of geocoded orders.

In [ ]:
import pandas as pd
import requests
from math import radians, cos, sin, sqrt, atan2

# Load order data
order_data2 = pd.read_excel(os.path.join(DATA_DIR, SYS_ORDER_FILE), sheet_name=None)
order_header2 = order_data['order_header']
order_line2 = order_data['order_line']

# Load geo data
geo_data2 = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# Standardize customer ID format in geo data
geo_data2['CUSTOMER_ID'] = geo_data2['CUSTOMER_ID'].str.replace('-', '')

# Merge order header and order line data on order_id
order_details2 = order_header2.merge(order_line, on='order_id')

# Calculate total full pallets per order
order_details2['total_full_pallets'] = order_details2.groupby('order_id')['full_pallets'].transform('sum')

# Function to geocode address using Google Maps API
def geocode_postcode(postcode, maps_key):
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": postcode, "key": maps_key}
    response = requests.get(base_url, params=params)
    geo_data = response.json()
    if geo_data['status'] == 'OK':
        location = geo_data['results'][0]['geometry']['location']
        return (location['lat'], location['lng'])
    else:
        return (None, None)

# Your Google Maps API key
maps_key = "<REDACTED_KEY>"

# Geocode each postcode in the order details
order_details2['lat_long'] = order_details2['postcode'].apply(lambda x: geocode_postcode(x, maps_key))

# Separate latitude and longitude into different columns
order_details2[['latitude', 'longitude']] = pd.DataFrame(order_details2['lat_long'].tolist(), index=order_details2.index, columns=['latitude', 'longitude'])

# Drop rows with missing latitude or longitude
order_details2 = order_details2.dropna(subset=['latitude', 'longitude'])

# Function to calculate haversine distance
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# Identify nearby consolidation opportunities
opportunities2 = []
for i, row3 in order_details2.iterrows():
    for j, row4 in order_details2.iterrows():
        if i >= j:
            continue
        distance2 = haversine_distance(row3['latitude'], row3['longitude'], row4['latitude'], row4['longitude'])
        if distance2 <= 48:  # 30 miles is approximately 48 kilometers
            combined_pallets2 = row3['total_full_pallets'] + row4['total_full_pallets']
            opportunities2.append((row3['order_id'], row4['order_id'], combined_pallets, distance))

# Convert opportunities to DataFrame for better visualization
opportunities_df2 = pd.DataFrame(opportunities2, columns=['Order 3', 'Order 4', 'Combined Pallets', 'Distance (km)'])

# Group by 'Order 1' and 'Order 2' to remove duplicate pairs
opportunities_df2 = opportunities_df2.groupby(['Order 3', 'Order 4']).agg({'Combined Pallets': 'first', 'Distance (km)': 'first'}).reset_index()

# Display the DataFrame
print("Consolidation Opportunities using Google Maps API:")
print(opportunities_df2)

# Save the opportunities DataFrame to an Excel file for easy review
opportunities_df2.to_excel(os.path.join(DATA_DIR, 'consolidation_opportunities_google_maps.xlsx'), index=False)

print("Consolidation opportunities have been saved to os.path.join(DATA_DIR, 'consolidation_opportunities_google_maps.xlsx').")

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of orders with consolidation opportunities
plt.figure(figsize=(10, 6))
plt.scatter(order_details2['longitude'], order_details2['latitude'], c='blue', label='Orders', alpha=0.5)
for index, row in opportunities_df.iterrows():
    order3 = order_details2[order_details2['order_id'] == row['Order 1']].iloc[0]
    order4 = order_details2[order_details2['order_id'] == row['Order 2']].iloc[0]
    plt.plot([order3['longitude'], order4['longitude']], [order3['latitude'], order4['latitude']], 'ro-', label='Consolidation Opportunity' if index == 0 else "", alpha=0.7)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Scatter Plot of Orders with Consolidation Opportunities')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming opportunities_df2 is already available

# Simplify x-axis labels
opportunities_df2['Order Pair'] = opportunities_df2['Order 3'].astype(str) + " & " + opportunities_df2['Order 4'].astype(str)

# Sort and filter data to show the top 20 opportunities
top_opportunities_df = opportunities_df2.sort_values(by='Combined Pallets', ascending=False).head(20)

# Create the bar plot
plt.figure(figsize=(14, 8))
sns.barplot(data=top_opportunities_df, x='Order Pair', y='Combined Pallets', hue='Order Pair', dodge=False, palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 Combined Pallets for Consolidation Opportunities')
plt.xlabel('Order Pairs')
plt.ylabel('Combined Pallets')
plt.legend().set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Histogram of distances between consolidation opportunities
plt.figure(figsize=(10, 6))
sns.histplot(opportunities_df2['Distance (km)'], bins=30, kde=True)
plt.title('Histogram of Distances Between Consolidation Opportunities')
plt.xlabel('Distance (km)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import pandas as pd

# Load the data
order_details2 = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# Print the column names to check for correct column names
print(order_details2.columns)


###### Original

In [ ]:
if 'latitude' in order_details2.columns and 'longitude' in order_details2.columns:
    m = folium.Map(location=[order_details2['latitude'].mean(), order_details2['longitude'].mean()], zoom_start=10)
else:
    m = folium.Map(location=[0, 0], zoom_start=10)

# Add consolidation opportunities to the map
for idx, row in opportunities_df2.iterrows():
    order3 = order_details2[order_details2['order_id'] == row['Order 3']].iloc[0]
    order4 = order_details2[order_details2['order_id'] == row['Order 4']].iloc[0]
    folium.PolyLine([(order1['latitude'], order1['longitude']), (order2['latitude'], order2['longitude'])], color='red', weight=2.5, opacity=0.8).add_to(m)

# Save and display the map
m.save(os.path.join(DATA_DIR, 'consolidation_opportunities_map.html'))
m

In [ ]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd
from pyproj import Proj, transform

# Load the data
order_details2 = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))
opportunities_df2 = pd.read_excel(os.path.join(DATA_DIR, 'consolidation_opportunities_google_maps.xlsx'))

# Convert coordinates from British National Grid to WGS84
proj_bng = Proj(init='epsg:27700')  # British National Grid
proj_wgs84 = Proj(init='epsg:4326')  # WGS84

def convert_to_wgs84(easting, northing):
    lon, lat = transform(proj_bng, proj_wgs84, easting, northing)
    return lat, lon

order_details2['latitude'], order_details2['longitude'] = zip(*order_details2.apply(lambda row: convert_to_wgs84(row['HOME EASTING'], row['HOME NORTHING']), axis=1))

# Create a map centered around the average location of the orders
m = folium.Map(location=[order_details2['latitude'].mean(), order_details2['longitude'].mean()], zoom_start=6)

# Add orders to the map
marker_cluster = MarkerCluster().add_to(m)
for idx, row in order_details2.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Customer ID: {row['CUSTOMER_ID']}, Postcode: {row['POSTCODE']}"
    ).add_to(marker_cluster)

# Add consolidation opportunities to the map
for idx, row in opportunities_df2.iterrows():
    order3 = order_details2.loc[order_details2['CUSTOMER_ID'] == row['Order 3']].iloc[0] if len(order_details2.loc[order_details['CUSTOMER_ID'] == row['Order 3']]) > 0 else None
    order4 = order_details2.loc[order_details2['CUSTOMER_ID'] == row['Order 4']].iloc[0] if len(order_details2.loc[order_details['CUSTOMER_ID'] == row['Order 4']]) > 0 else None
    if order1 is not None and order2 is not None:
        folium.PolyLine(
            locations=[(order1['latitude'], order1['longitude']), (order2['latitude'], order2['longitude'])],
            color='red',
            weight=2.5,
            opacity=0.8
        ).add_to(m)

# Save and display the map
m.save(os.path.join(DATA_DIR, 'consolidation_opportunities_map_improved.html'))


##### Merge and Calculate Pallet Sums:
###### Merge the filtered order_header with order_line to calculate the sum of full pallets per consignment.
###### Identify consignments with less than 45 full pallets.